In [2]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 667, done.
remote: Counting objects: 100% (233/233), done.
remote: Compressing objects: 100% (136/136), done.
remote: Total 667 (delta 174), reused 100 (delta 97), pack-reused 434 (from 3)
Receiving objects: 100% (667/667), 220.60 KiB | 14.71 MiB/s, done.
Resolving deltas: 100% (348/348), done.
Installing RAPIDS remaining 26.02 libraries
Using Python 3.12.13 environment at: /usr
Resolved 180 packages in 9.65s
Prepared 10 packages in 1.24s
Uninstalled 4 packages in 196ms
Installed 10 packages in 44ms
 - bokeh==3.8.2
 + bokeh==3.6.3
 + cugraph-cu12==26.2.0
 + cuxfilter-cu12==26.2.0
 + datashader==0.19.1
 - holoviews==1.22.1
 + holoviews==1.20.2
 + jupyter-server-proxy==4.5.0
 - panel==1.9.3
 + panel==1.7.5
 + pyct==0.6.0
 - shapely==2.1.2
 + shapely==2.0.7
 + simpervisor==1.0.0

        ***********************************************************************
        The pip install of RAPIDS is complete.

        Please do 

In [3]:
import xgboost as xgb
import pandas as pd
import numpy as np
import cudf
import cuml
import seaborn as sns

from tensorflow import keras
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, balanced_accuracy_score
from cuml.ensemble import RandomForestClassifier as cuRF

In [5]:
df = pd.read_csv("dataset_completo_40036_v2.csv")

COLS_ELIMINAR = ['Tipo', 'L', 'B', 'A']
X = df.drop(COLS_ELIMINAR, axis=1)
y = df['Tipo']

class_counts = y.value_counts()
clases_validas = class_counts[class_counts >= 10].index
mask = y.isin(clases_validas)

X_filt = X[mask].copy()
y_filt = y[mask].copy()

clases_eliminadas = class_counts[class_counts < 10].index.tolist()

nan_cols = X_filt.columns[X_filt.isna().any()].tolist()


le = LabelEncoder()
y_enc = le.fit_transform(y_filt).astype(np.int32)


X_train, X_test, y_train, y_test = train_test_split(
    X_filt, y_enc,
    test_size=0.2,
    random_state=42,
    stratify=y_enc
)

imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp).astype(np.float32)
X_test_scaled  = scaler.transform(X_test_imp).astype(np.float32)


In [6]:
mlp = MLPClassifier(
    hidden_layer_sizes=(100,50,50),
    activation='tanh',
    solver='sgd',
    max_iter=1000,
    alpha = 0.001,
    random_state=42,
    learning_rate='adaptive',
)

from sklearn.utils.class_weight import compute_sample_weight
sw = compute_sample_weight('balanced', y_train)
mlp.fit(X_train_scaled, y_train)
predicciones = mlp.predict(X_test_scaled)

precision = accuracy_score(y_test, predicciones)
print(f"La precisión de la red es de: {precision * 100}%")

La precisión de la red es de: 99.69543147208122%


In [7]:
clases    = le.classes_
etiquetas = [str(c) for c in clases]

# Predicciones
y_pred = mlp.predict(X_test_scaled)

# Precision / recall / f1 por clase
rep = classification_report(y_test, y_pred, target_names=etiquetas,
                            output_dict=True, zero_division=0)
print(classification_report(y_test, y_pred, target_names=etiquetas, zero_division=0))

prec = [rep[l]["precision"] for l in etiquetas]
rec  = [rep[l]["recall"]    for l in etiquetas]
f1   = [rep[l]["f1-score"]  for l in etiquetas]


                   precision    recall  f1-score   support

   Miel de Bosque       0.99      1.00      1.00      1352
Miel de milflores       1.00      0.99      1.00      1603

         accuracy                           1.00      2955
        macro avg       1.00      1.00      1.00      2955
     weighted avg       1.00      1.00      1.00      2955



In [8]:
model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss',
    tree_method='hist',
    device='cuda',
    random_state=42
)

param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200],
    'subsample': [0.8, 1.0]
}

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv = 5, scoring='accuracy')

grid_search.fit(X_train_scaled, y_train)

print(f"Mejor configuración: {grid_search.best_params_}")
print(f"Mejor score de validación: {grid_search.best_score_:.4f}")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:03:29] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [20:03:29] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:03:30] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain

Mejor configuración: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 200, 'subsample': 0.8}
Mejor score de validación: 0.9981


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:04:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [9]:
clases    = le.classes_
etiquetas = [str(c) for c in clases]

# Predicciones
best_xgb = grid_search.best_estimator_
y_pred   = best_xgb.predict(X_test_scaled)

# precision / recall / f1 por clase
rep = classification_report(y_test, y_pred, target_names=etiquetas,
                            output_dict=True, zero_division=0)
print(classification_report(y_test, y_pred, target_names=etiquetas, zero_division=0))

prec = [rep[l]["precision"] for l in etiquetas]
rec  = [rep[l]["recall"]    for l in etiquetas]
f1   = [rep[l]["f1-score"]  for l in etiquetas]


                   precision    recall  f1-score   support

   Miel de Bosque       1.00      1.00      1.00      1352
Miel de milflores       1.00      1.00      1.00      1603

         accuracy                           1.00      2955
        macro avg       1.00      1.00      1.00      2955
     weighted avg       1.00      1.00      1.00      2955



In [10]:
import tensorflow as tf
import numpy as np

#Generar predicciones "suaves" del XGBoost sobre TODO el train
best_model = grid_search.best_estimator_
soft_labels = best_model.predict_proba(X_train_scaled)  # shape (n_samples, 7)

n_features = X_train_scaled.shape[1]
n_clases = soft_labels.shape[1]

#Definir red Keras
model_keras = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(n_features,)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(n_clases, activation='softmax')
])

model_keras.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

#Entrenar imitando al XGBoost
model_keras.fit(
    X_train_scaled, soft_labels,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

#Evaluar vs el XGBoost original
y_pred_keras = np.argmax(model_keras.predict(X_test_scaled), axis=1)
y_pred_xgb   = best_model.predict(X_test_scaled)

print(f"Accuracy Keras:   {accuracy_score(y_test, y_pred_keras):.4f}")
print(f"Accuracy XGBoost: {accuracy_score(y_test, y_pred_xgb):.4f}")

#Guardar en formato Keras
model_keras.save('mejor_modelo.keras')

import json
with open('clases.json', 'w', encoding='utf-8') as f:
    json.dump(le.classes_.tolist(), f, ensure_ascii=False)
print(f"Clases guardadas ({len(le.classes_)}): {le.classes_.tolist()}")

Epoch 1/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - accuracy: 0.9818 - loss: 0.0729 - val_accuracy: 0.9949 - val_loss: 0.0170
Epoch 2/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9948 - loss: 0.0178 - val_accuracy: 0.9958 - val_loss: 0.0165
Epoch 3/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9982 - loss: 0.0103 - val_accuracy: 0.9941 - val_loss: 0.0186
Epoch 4/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9976 - loss: 0.0107 - val_accuracy: 0.9992 - val_loss: 0.0102
Epoch 5/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9992 - loss: 0.0081 - val_accuracy: 0.9975 - val_loss: 0.0132
Epoch 6/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9988 - loss: 0.0086 - val_accuracy: 0.9983 - val_loss: 0.0093
Epoch 7/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9987 - loss: 0.0086 - val_accuracy: 0.9975 - val_loss: 0.0108
Epoch 8/50
333/333 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9992 - loss: 0.0073 - val_accuracy: 

In [11]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

clases    = le.classes_
etiquetas = [str(c) for c in clases]

# Predicciones
y_pred = np.argmax(model_keras.predict(X_test_scaled), axis=1)

# Informe con precision / recall / f1 por clase
rep = classification_report(y_test, y_pred, target_names=etiquetas,
                            output_dict=True, zero_division=0)
print(classification_report(y_test, y_pred, target_names=etiquetas, zero_division=0))

prec = [rep[l]["precision"] for l in etiquetas]
rec  = [rep[l]["recall"]    for l in etiquetas]
f1   = [rep[l]["f1-score"]  for l in etiquetas]


93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
                   precision    recall  f1-score   support

   Miel de Bosque       1.00      1.00      1.00      1352
Miel de milflores       1.00      1.00      1.00      1603

         accuracy                           1.00      2955
        macro avg       1.00      1.00      1.00      2955
     weighted avg       1.00      1.00      1.00      2955



In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

#Constantes
ARCHIVO_ORIGEN = "Primeras muestras.csv"
ARCHIVO_SALIDA = "dataset_completo_40036_v3.csv"
COLUMNA_CLASE  = "Tipo"

# Los 18 canales espectrales
CANALES = ["A_410", "B_435", "C_460", "D_485", "E_510", "F_535",
           "G_560", "H_585", "R_610", "I_645", "S_680", "J_705",
           "T_730", "U_760", "V_810", "W_860", "K_900", "L_940"]

N_SINTETICAS   = 953        #Esto nos genera unas 40036 muestras
SIGMA          = 0.01
RUIDO_RELATIVO = False      # Lo ponemos a false para que no mantengga la estructura estadística
SEMILLA        = 42

rng = np.random.default_rng(SEMILLA)

df = pd.read_csv(Path(ARCHIVO_ORIGEN), encoding="latin-1", sep=";", decimal=",")

X = df[CANALES].to_numpy(dtype=np.float64)
y = df[COLUMNA_CLASE].to_numpy()


filas_sinteticas, etiquetas_sinteticas = [], []
for firma, etiqueta in zip(X, y):
    sigma_vec = SIGMA * np.abs(firma) if RUIDO_RELATIVO else SIGMA
    ruido = rng.normal(loc=0.0, scale=sigma_vec, size=(N_SINTETICAS, X.shape[1]))
    filas_sinteticas.append(firma + ruido)
    etiquetas_sinteticas.extend([etiqueta] * N_SINTETICAS)

X_sint = np.vstack(filas_sinteticas)

X_final = np.vstack([X, X_sint])
y_final = np.concatenate([y, etiquetas_sinteticas])

df_final = pd.DataFrame(X_final, columns=CANALES)
df_final[COLUMNA_CLASE] = y_final
df_final = df_final.sample(frac=1, random_state=SEMILLA).reset_index(drop=True)

df_final.to_csv(Path(ARCHIVO_SALIDA), index=False, encoding="utf-8-sig")

print(f"Firmas reales:       {len(df)}")
print(f"Muestras sintéticas: {len(X_sint)}")
print(f"Total dataset:       {len(df_final)}")
print(df_final[COLUMNA_CLASE].value_counts())